# Yoruba HealthQA -- Run the App on Kaggle's GPU (public link)

Runs the real app with the fine-tuned YorubaLlama model on Kaggle's free GPU and prints a public `https://....gradio.live` link anyone can open in a browser. Nothing runs on your laptop.

## Before you start
1. **Upload your trained adapter as a Kaggle Dataset.** From your laptop's `models/single_run/` folder, upload ONLY these files: `adapter_config.json`, `adapter_model.safetensors`, `tokenizer.json`, `tokenizer_config.json`. Do NOT upload the `checkpoint-33` folder (large training leftovers, not needed). kaggle.com/datasets -> New Dataset.
2. In this notebook: **Add Input** -> attach that adapter dataset.
3. **Add-ons -> Secrets**: attach your `HF_TOKEN` secret (same one as before).
4. Settings: **GPU** on (T4), **Internet** on.
5. Click **Save Version -> Save & Run All (Commit)**. This keeps running in the background even if you close the tab (up to about 12 hours).
6. Open that version's **Logs** tab and look for a line like `Running on public URL: https://xxxx.gradio.live`. That is the link to share.

**Limits to know:** the link only works while this Kaggle run is alive (max about 12 hours, and it uses your weekly free GPU hours). Anyone with the link can use it. For a permanent link you would need a paid host.

In [ ]:
!nvidia-smi

In [ ]:
import shutil

%cd /kaggle/working
shutil.rmtree("Yoruba-HealthQA", ignore_errors=True)
!git clone https://github.com/ayoolaeni/Yoruba-HealthQA.git
%cd Yoruba-HealthQA

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate gradio pyyaml python-dotenv rich

In [ ]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    _token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    import getpass
    _token = getpass.getpass("Paste your Hugging Face access token: ")
login(token=_token)

In [ ]:
# Find the attached adapter at any depth under /kaggle/input/.
from pathlib import Path

kaggle_input = Path("/kaggle/input")
adapter_dir = None
for d in [kaggle_input] + [p for p in kaggle_input.rglob("*") if p.is_dir()]:
    if (d / "adapter_config.json").exists() and (d / "adapter_model.safetensors").exists():
        adapter_dir = d
        break
assert adapter_dir is not None, (
    "No adapter_config.json + adapter_model.safetensors found under /kaggle/input/. "
    "Attach your adapter dataset via 'Add Input' first."
)
print("Adapter found at:", adapter_dir)

In [ ]:
# Build the real app (same code as app/app.py) with the fine-tuned model.
# On a GPU the app loads the base model in 4-bit automatically.
import os
import sys

os.environ["YHQA_BASE_MODEL"] = "Jacaranda/YorubaLlama"
os.environ["YHQA_ADAPTER_PATH"] = str(adapter_dir)
os.environ["YHQA_MAX_NEW_TOKENS"] = "150"
sys.path.insert(0, os.getcwd())

import gradio as gr
from app.app import build_demo

demo = build_demo()
demo.launch(share=True, theme=gr.themes.Soft(),
            css=".gradio-container {max-width: 640px !important; margin: 0 auto !important}")

In [ ]:
# Keep the app alive. Without this, a committed run would finish right
# after the previous cell and the public link would stop working.
demo.block_thread()